In [23]:
import refinitiv.data as rd
import pandas as pd
import numpy as np

rd.open_session()

<refinitiv.data.session.Definition object at 0x11190dc40 {name='workspace'}>

In [24]:
universe = pd.read_csv("../../data/stoxx600_universe.csv")
STOCK = universe["RIC"].sample(30, random_state=123).tolist()
PARAMS = {"SDate": "2024-01-01", "EDate": "2025-12-31", "Frq": "D", "Curn": "EUR"}

df = rd.get_data(
    universe=STOCK,
    fields=[
        "TR.PriceClose.date",
        "TR.PriceClose",
        "TR.CompanyMarketCap",
        "TR.PriceToCFPerShare",
        "TR.F.NetCashFlowOp",
        "TR.CFPSActValue",
        "TR.F.NetCFOpPerShr",
    ],
    parameters={**PARAMS, "Period": "LTM"}
)

print(f"Sarakkeet: {df.columns.tolist()}")
print(f"{len(df)} riviä")
df

Sarakkeet: ['Instrument', 'Date', 'Price Close', 'Company Market Cap', 'Price To Cash Flow Per Share (Daily Time Series Ratio)', 'Net Cash Flow from Operating Activities', 'Cash Flow Per Share - Actual', 'Cash Flow from Operations per Share']
15161 riviä


/opt/anaconda3/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
/opt/anaconda3/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
/opt/anaconda3/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to

,Instrument,Date,Price Close,Company Market Cap,Price To Cash Flow Per Share (Daily Time Series Ratio),Net Cash Flow from Operating Activities,Cash Flow Per Share - Actual,Cash Flow from Operations per Share
0,NEXII.MI,2024-01-02,7.312,9595907053.42395,24.624463,393707000.0,<NA>,0.28489
1,NEXII.MI,2024-01-03,7.092,9307189937.48398,23.883574,393707000.0,<NA>,0.28489
2,NEXII.MI,2024-01-04,7.062,9267819421.674,23.782544,393707000.0,<NA>,0.28489
3,NEXII.MI,2024-01-05,7.096,9312439339.591921,23.897045,393707000.0,<NA>,0.28489
4,NEXII.MI,2024-01-08,7.26,9527664826.019911,24.449344,393707000.0,<NA>,0.28489
...,...,...,...,...,...,...,...,...
15156,ALEP.WA,2025-12-19,7.353706,7772167202.43159,14.076009,452000382.011567,<NA>,<NA>
15157,ALEP.WA,2025-12-22,7.347006,7765086237.48852,14.105624,452000382.011567,<NA>,<NA>
15158,ALEP.WA,2025-12-23,7.257912,7670922264.07649,13.952995,452000382.011567,<NA>,<NA>
15159,ALEP.WA,2025-12-29,7.386953,7807306519.68985,14.212691,452000382.011567,<NA>,<NA>


In [25]:
# Tyyppimuunnokset
print("Sarakkeet:", df.columns.tolist())
for c in df.columns[2:]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

Price = df["Price Close"]
MktCap = df["Company Market Cap"]
OpCF = df["Net Cash Flow from Operating Activities"]
OpCFPS = df["Cash Flow from Operations per Share"]

# Itselasketut P/CF
out = df[["Instrument", "Date", "Price To Cash Flow Per Share (Daily Time Series Ratio)"]].copy()
out.columns = ["Instrument", "Date", "REF_PCF"]
out["PCF_MktCap_div_OpCF"]   = MktCap / OpCF
out["PCF_Price_div_OpCFPS"]  = Price / OpCFPS
out

Sarakkeet: ['Instrument', 'Date', 'Price Close', 'Company Market Cap', 'Price To Cash Flow Per Share (Daily Time Series Ratio)', 'Net Cash Flow from Operating Activities', 'Cash Flow Per Share - Actual', 'Cash Flow from Operations per Share']


,Instrument,Date,REF_PCF,PCF_MktCap_div_OpCF,PCF_Price_div_OpCFPS
0,NEXII.MI,2024-01-02,24.624463,24.373219,25.666003
1,NEXII.MI,2024-01-03,23.883574,23.639889,24.893776
2,NEXII.MI,2024-01-04,23.782544,23.53989,24.788472
3,NEXII.MI,2024-01-05,23.897045,23.653223,24.907816
4,NEXII.MI,2024-01-08,24.449344,24.199887,25.483476
...,...,...,...,...,...
15156,ALEP.WA,2025-12-19,14.076009,17.195046,<NA>
15157,ALEP.WA,2025-12-22,14.105624,17.17938,<NA>
15158,ALEP.WA,2025-12-23,13.952995,16.971053,<NA>
15159,ALEP.WA,2025-12-29,14.212691,17.272787,<NA>
